# Roadside Parking Detection — Geometry-Based Occupancy Model

This notebook trains an object-detection model (YOLOv8, via `ultralytics`) on the
**Roadside Parking** dataset from Roboflow Universe:

https://universe.roboflow.com/3883zn-gmail-com/roadside-parking/dataset/12

After training the detector, we add a **geometry-based post-processing layer** that:
- Defines parking-slot polygons (from dataset annotations or manually drawn ROIs)
- Computes IoU / centroid-in-polygon overlap between detected vehicles and each slot
- Classifies each slot as **occupied** / **free** based on geometric overlap, not just raw detection

## Pipeline
1. Environment setup
2. Download dataset from Roboflow (requires your own API key)
3. Explore dataset (classes, image/label counts, sample visualization)
4. Train YOLOv8 detector
5. Evaluate (mAP, PR curve, confusion matrix)
6. Geometry-based parking-slot occupancy logic
7. Inference demo on a sample image/video


## 1. Environment Setup

In [ ]:
!pip install -q ultralytics roboflow opencv-python shapely matplotlib pandas seaborn


In [ ]:
import os
import cv2
import glob
import yaml
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Polygon, box

from ultralytics import YOLO

SEED = 42
np.random.seed(SEED)

WORK_DIR = Path("./roadside_parking")
WORK_DIR.mkdir(exist_ok=True)
print("Working dir:", WORK_DIR.resolve())


## 2. Download the Dataset from Roboflow

Roboflow Universe datasets are pulled via the `roboflow` Python SDK. You need a free
Roboflow account and API key (Settings → Roboflow API → Private API Key).

> Replace `YOUR_API_KEY` below. Do **not** commit your real key to version control —
> use an environment variable instead (`os.environ["ROBOFLOW_API_KEY"]`).

The dataset URL you gave (`.../roadside-parking/dataset/12`) tells us:
- workspace: `3883zn-gmail-com`
- project: `roadside-parking`
- version: `12`

We export in **YOLOv8** format, which is what `ultralytics` expects (images +
`.txt` label files + a `data.yaml` describing classes/splits).


In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_API_KEY")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("3883zn-gmail-com").project("roadside-parking")
version = project.version(12)

dataset = version.download("yolov8", location=str(WORK_DIR / "dataset"))
print("Dataset downloaded to:", dataset.location)


In [ ]:
DATA_YAML = Path(dataset.location) / "data.yaml"

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print(json.dumps(data_cfg, indent=2))
CLASS_NAMES = data_cfg["names"]
NUM_CLASSES = data_cfg["nc"] if "nc" in data_cfg else len(CLASS_NAMES)
print(f"\nClasses ({NUM_CLASSES}): {CLASS_NAMES}")


## 3. Explore the Dataset

In [ ]:
def count_split(split_dir):
    img_dir = Path(dataset.location) / split_dir / "images"
    lbl_dir = Path(dataset.location) / split_dir / "labels"
    n_img = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    return n_img, n_lbl

rows = []
for split in ["train", "valid", "test"]:
    n_img, n_lbl = count_split(split)
    rows.append({"split": split, "images": n_img, "labels": n_lbl})

df_counts = pd.DataFrame(rows)
df_counts


In [ ]:
# Class distribution across all label files
from collections import Counter

class_counter = Counter()
for split in ["train", "valid", "test"]:
    lbl_dir = Path(dataset.location) / split / "labels"
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                if line.strip():
                    cls_id = int(line.split()[0])
                    class_counter[CLASS_NAMES[cls_id]] += 1

plt.figure(figsize=(8, 4))
plt.bar(class_counter.keys(), class_counter.values(), color="steelblue")
plt.title("Class distribution")
plt.ylabel("Instance count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
def yolo_to_xyxy(box_norm, img_w, img_h):
    xc, yc, w, h = box_norm
    x1 = (xc - w / 2) * img_w
    y1 = (yc - h / 2) * img_h
    x2 = (xc + w / 2) * img_w
    y2 = (yc + h / 2) * img_h
    return [x1, y1, x2, y2]

def visualize_sample(split="train", n=4):
    img_dir = Path(dataset.location) / split / "images"
    lbl_dir = Path(dataset.location) / split / "labels"
    img_paths = sorted(img_dir.glob("*.*"))[:n]

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, img_path in zip(axes, img_paths):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    cls_id = int(parts[0])
                    box_norm = list(map(float, parts[1:5]))
                    x1, y1, x2, y2 = map(int, yolo_to_xyxy(box_norm, w, h))
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img, CLASS_NAMES[cls_id], (x1, max(y1 - 5, 0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        ax.imshow(img)
        ax.axis("off")
        ax.set_title(img_path.name, fontsize=9)

    plt.tight_layout()
    plt.show()

visualize_sample("train", n=4)


## 4. Train YOLOv8 Detector

We fine-tune a pretrained `yolov8n.pt` (nano — fast, good baseline). Swap to
`yolov8s.pt` / `yolov8m.pt` for higher accuracy if you have GPU budget.


In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project=str(WORK_DIR / "runs"),
    name="roadside_parking_yolov8n",
    seed=SEED,
    device=0,          # set to "cpu" if no GPU available
    plots=True,
)


## 5. Evaluate the Model

In [ ]:
metrics = model.val(data=str(DATA_YAML), split="test")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Per-class mAP50:", dict(zip(CLASS_NAMES, metrics.box.maps)))


In [ ]:
best_weights = WORK_DIR / "runs" / "roadside_parking_yolov8n" / "weights" / "best.pt"
print("Best weights at:", best_weights)

# Visualize training curves already saved by ultralytics
run_dir = WORK_DIR / "runs" / "roadside_parking_yolov8n"
for img_name in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = run_dir / img_name
    if p.exists():
        img = plt.imread(p)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_name)
        plt.show()


## 6. Geometry-Based Parking-Slot Occupancy

Raw detections tell you "there's a car here" but not "which parking slot is
occupied." We close that gap with pure geometry:

1. **Define parking slots** as polygons — either:
   - Loaded from a slot-map JSON you draw once per camera view, or
   - Derived from dataset annotations if the dataset includes a `parking-space` class
2. **Match detections to slots** using IoU between the detection box and each slot
   polygon (or centroid-in-polygon containment as a simpler alternative).
3. **Occupancy decision**: a slot is `occupied` if the best-matching detection's IoU
   (or overlap ratio) exceeds a threshold, else `free`.

This is the same idea used in real parking-lot camera systems: detector for "what,"
geometry for "where relative to a fixed slot."


In [ ]:
def polygon_iou(poly_a, poly_b):
    """IoU between two shapely polygons."""
    if not poly_a.is_valid or not poly_b.is_valid:
        return 0.0
    inter = poly_a.intersection(poly_b).area
    union = poly_a.union(poly_b).area
    return inter / union if union > 0 else 0.0


def overlap_ratio(det_poly, slot_poly):
    """Fraction of the SLOT area covered by the detection (robust to partial cars)."""
    if not det_poly.is_valid or not slot_poly.is_valid:
        return 0.0
    inter = det_poly.intersection(slot_poly).area
    return inter / slot_poly.area if slot_poly.area > 0 else 0.0


class ParkingSlot:
    def __init__(self, slot_id, points):
        """points: list of (x, y) pixel coordinates defining the slot polygon."""
        self.slot_id = slot_id
        self.polygon = Polygon(points)

    def check_occupancy(self, detections, iou_thresh=0.15, overlap_thresh=0.35):
        """
        detections: list of dicts {"box": [x1,y1,x2,y2], "cls": str, "conf": float}
        Returns (is_occupied: bool, best_match: dict or None, score: float)
        """
        best_score = 0.0
        best_match = None
        for det in detections:
            x1, y1, x2, y2 = det["box"]
            det_poly = box(x1, y1, x2, y2)
            score = max(
                polygon_iou(det_poly, self.polygon),
                overlap_ratio(det_poly, self.polygon),
            )
            if score > best_score:
                best_score = score
                best_match = det

        is_occupied = best_score >= min(iou_thresh, overlap_thresh) if best_match else False
        # Use overlap_thresh as the primary rule since car boxes rarely match slot shape exactly
        is_occupied = best_match is not None and (
            overlap_ratio(box(*best_match["box"]), self.polygon) >= overlap_thresh
            or polygon_iou(box(*best_match["box"]), self.polygon) >= iou_thresh
        )
        return is_occupied, best_match, best_score


In [ ]:
# Example slot map — replace with real coordinates for your camera view.
# You can create this once with a simple click-to-draw tool (cv2.setMouseCallback)
# on a reference frame from your dataset/video.

SLOT_MAP_EXAMPLE = [
    {"slot_id": "A1", "points": [(50, 300), (180, 300), (180, 420), (50, 420)]},
    {"slot_id": "A2", "points": [(190, 300), (320, 300), (320, 420), (190, 420)]},
    {"slot_id": "A3", "points": [(330, 300), (460, 300), (460, 420), (330, 420)]},
]

parking_slots = [ParkingSlot(s["slot_id"], s["points"]) for s in SLOT_MAP_EXAMPLE]

def draw_slot_map(image_path, slots, save_path=None):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for slot in slots:
        pts = np.array(slot.polygon.exterior.coords, dtype=np.int32)
        cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 255), thickness=2)
        cx, cy = slot.polygon.centroid.coords[0]
        cv2.putText(img, slot.slot_id, (int(cx) - 10, int(cy)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Parking slot map")
    plt.show()
    if save_path:
        cv2.imwrite(str(save_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


## 7. Inference: Detector + Geometry Occupancy

In [ ]:
def run_inference_with_occupancy(image_path, model, slots, conf_thres=0.35):
    result = model.predict(source=str(image_path), conf=conf_thres, verbose=False)[0]

    detections = []
    for b in result.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cls_id = int(b.cls[0].item())
        conf = float(b.conf[0].item())
        detections.append({
            "box": [x1, y1, x2, y2],
            "cls": CLASS_NAMES[cls_id],
            "conf": conf,
        })

    occupancy = {}
    for slot in slots:
        is_occ, match, score = slot.check_occupancy(detections)
        occupancy[slot.slot_id] = {
            "occupied": is_occ,
            "score": round(score, 3),
            "matched_class": match["cls"] if match else None,
        }

    return detections, occupancy


def visualize_occupancy(image_path, detections, slots, occupancy):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for det in detections:
        x1, y1, x2, y2 = map(int, det["box"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, f'{det["cls"]} {det["conf"]:.2f}', (x1, max(y1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    for slot in slots:
        pts = np.array(slot.polygon.exterior.coords, dtype=np.int32)
        color = (255, 0, 0) if occupancy[slot.slot_id]["occupied"] else (0, 200, 0)
        label = "OCCUPIED" if occupancy[slot.slot_id]["occupied"] else "FREE"
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=3)
        cx, cy = slot.polygon.centroid.coords[0]
        cv2.putText(img, f"{slot.slot_id}: {label}", (int(cx) - 40, int(cy)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    plt.figure(figsize=(10, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


# --- Run on a sample test image ---
test_images = sorted((Path(dataset.location) / "test" / "images").glob("*.*"))
if test_images:
    sample_img = test_images[0]
    dets, occ = run_inference_with_occupancy(sample_img, model, parking_slots)
    print(json.dumps(occ, indent=2))
    visualize_occupancy(sample_img, dets, parking_slots, occ)
else:
    print("No test images found — check dataset split names in data.yaml")


## 8. Export the Trained Model

Export to ONNX (or TFLite/CoreML) for deployment on edge devices / parking cameras.


In [ ]:
model.export(format="onnx", imgsz=640)
print("Exported ONNX model next to best.pt")


## Notes / Next Steps

- **Slot map**: replace `SLOT_MAP_EXAMPLE` with real polygon coordinates for your
  camera view. A quick way to get these: open one reference frame, click 4 corners
  per slot with `cv2.setMouseCallback`, and save to a JSON file.
- **Thresholds**: tune `iou_thresh` / `overlap_thresh` in `ParkingSlot.check_occupancy`
  against a few labeled example frames (occupied vs free) to calibrate.
- **Temporal smoothing**: for video, smooth `occupied` flags over N frames (majority
  vote) to avoid flicker from momentary occlusion/misses.
- **Bird's-eye view**: for angled camera views, consider a homography transform to a
  top-down view before geometry matching — this makes IoU-based occupancy far more
  reliable than raw image-plane IoU.
- **Class filtering**: if the dataset has multiple vehicle classes (car, truck, bike),
  filter `detections` by class before matching to slots if only certain vehicle types
  should count as "occupying" a spot.
